In [ ]:
from blackjack.blackjack_round import BJRound, BJStage, BJRules
from blackjack.actions import PlayerAction, DealerAction
from blackjack.cards import Card, Rank
import numpy as np
import time
import logging
from datetime import datetime
import os
from logging import FileHandler
import tqdm
from collections import deque
from blackjack.shoe import ProbabilisticRankShoe

In [2]:
shoe = ProbabilisticRankShoe(1)

In [3]:
for r in range(2, 9):
    for i in range(4):
        shoe.burn_rank_value(r)

In [4]:
print(str(shoe))

ProbabilisticRankShoe
  p( 9) = 16.67%
  p(10) = 66.67%
  p(11) = 16.67%


In [5]:
shoe.lock_dealer_card_not_ten()
print(str(shoe))

ProbabilisticRankShoe
  p( 9|D≠10) = 15.22%
  p(10|D≠10) = 69.57%
  p(11|D≠10) = 15.22%


In [6]:
shoe.lock_dealer_card_not_ace()
print(str(shoe))

ProbabilisticRankShoe
  p( 9|D≠11) = 16.52%
  p(10|D≠11) = 66.09%
  p(11|D≠11) = 17.39%


In [7]:
# Nicely formatted probabilities for the blackjack-valuation scenario
# Deck after burning ranks 2–8: 24 cards total
# Counts: 9 → 4 cards, 10 → 16 cards (10/J/Q/K), 11 → 4 cards (Aces)

def line():
    print("-" * 64)

print("Blackjack deck after burning 2–8 (24 cards total)")
print("Counts: 9×4, 10×16 (10/J/Q/K), 11×4 (Aces)")
line()

print("P(first card has value 9/10/11):")
print(f"  • 9  = 4/24 = 1/6   ≈ {4/24 * 100}%")
print(f"  • 10 = 16/24 = 2/3  ≈ {16/24 * 100}%")
print(f"  • 11 = 4/24 = 1/6   ≈ {4/24 * 100}%")
line()

print("Given the second card is NOT 11:")
print(f"  • 9  = 19/115 ≈ {19/115 * 100}%")
print(f"  • 10 = 76/115 ≈ {76/115 * 100}%")
print(f"  • 11 = 4/23   ≈ {4/23 * 100}%")
line()

print("Given the second card is NOT 10:")
print(f"  • 9  = 7/46   ≈ {7/46 * 100}%")
print(f"  • 10 = 16/23  ≈ {16/23 * 100}%")
print(f"  • 11 = 7/46   ≈ {7/46 * 100}%")


Blackjack deck after burning 2–8 (24 cards total)
Counts: 9×4, 10×16 (10/J/Q/K), 11×4 (Aces)
----------------------------------------------------------------
P(first card has value 9/10/11):
  • 9  = 4/24 = 1/6   ≈ 16.666666666666664%
  • 10 = 16/24 = 2/3  ≈ 66.66666666666666%
  • 11 = 4/24 = 1/6   ≈ 16.666666666666664%
----------------------------------------------------------------
Given the second card is NOT 11:
  • 9  = 19/115 ≈ 16.52173913043478%
  • 10 = 76/115 ≈ 66.08695652173913%
  • 11 = 4/23   ≈ 17.391304347826086%
----------------------------------------------------------------
Given the second card is NOT 10:
  • 9  = 7/46   ≈ 15.217391304347828%
  • 10 = 16/23  ≈ 69.56521739130434%
  • 11 = 7/46   ≈ 15.217391304347828%


In [34]:
shoe = ProbabilisticRankShoe(1)

for r in range(2, 8):
    for i in range(4):
        shoe.burn_rank_value(r)

for i in range(4 * 4 - 4):
    shoe.burn_rank_value(10)

for r in range(8, 12):
    for i in range(2):
        shoe.burn_rank_value(r)

In [35]:
print(shoe)

ProbabilisticRankShoe
  p( 8) = 25.00%
  p( 9) = 25.00%
  p(10) = 25.00%
  p(11) = 25.00%


In [41]:
shoe.lock_dealer_card_not_ace()
print(str(shoe))

ProbabilisticRankShoe
  p( 8|D≠11) = 23.81%
  p( 9|D≠11) = 23.81%
  p(10|D≠11) = 23.81%
  p(11|D≠11) = 28.57%


In [50]:
shoe.lock_dealer_card_not_ten()
print(str(shoe))

probs = shoe.get_rank_value_probabilities({10, 11, 9})
for rv, p in probs.items():
    if p > 0:
        print(f"Rank value {rv}: probability {p*100:.2f}%")

ProbabilisticRankShoe
  p( 8|D≠10) = 23.81%
  p( 9|D≠10) = 23.81%
  p(10|D≠10) = 28.57%
  p(11|D≠10) = 23.81%
Rank value 9: probability 31.25%
Rank value 10: probability 37.50%
Rank value 11: probability 31.25%


In [48]:
shoe.lock_dealer_card_not_ace()
print(str(shoe))
probs = shoe.get_rank_value_probabilities({8, 11})
for rv, p in probs.items():
    if p > 0:
        print(f"Rank value {rv}: probability {p*100:.2f}%")

ProbabilisticRankShoe
  p( 8|D≠11) = 23.81%
  p( 9|D≠11) = 23.81%
  p(10|D≠11) = 23.81%
  p(11|D≠11) = 28.57%
Rank value 8: probability 45.45%
Rank value 11: probability 54.55%


In [40]:
for r in range(2, 9):
    for i in range(4):
        shoe.burn_rank_value(r)

RuntimeError: Card count for rank value 2 is too low